In [30]:
import os
import time
import getpass

import pandas as pd
from sqlalchemy import create_engine, text, inspect

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print("pandas", pd.__version__)

pandas 3.0.5


In [31]:
DB_HOST = "10.100.71.21"
DB_PORT = 5432
DB_NAME = input("Database name: ").strip()
DB_USER = input("User name: ").strip()
DB_PASS = getpass.getpass("Password: ")

# Folder that contains the 12 CSV files
DATA_DIR = "data"

# Schema where the DDL was run ("public" unless you created your own schema)
SCHEMA = "public"

In [32]:
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

PostgreSQL 10.20 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 4.8.5 20150623 (Red Hat 4.8.5-44), 64-bit


In [33]:
LOAD_ORDER = [            # parents first, children after
    "categories", "suppliers", "stores", "promotions", "customers",
    "employees", "products", "orders",
    "order_items", "payments", "shipments", "returns",
]

existing = set(inspect(engine).get_table_names(schema=SCHEMA))
missing = [t for t in LOAD_ORDER if t not in existing]
print("Tables found  :", sorted(existing))
print("Tables missing:", missing if missing else "none")
assert not missing, "Run the Milestone 2 DDL first."

Tables found  : ['categories', 'customers', 'department', 'employees', 'order_items', 'orders', 'payments', 'products', 'program', 'promotions', 'returns', 'shipments', 'stores', 'student', 'suppliers']
Tables missing: none


In [34]:
db_columns = {
    t: [c["name"] for c in inspect(engine).get_columns(t, schema=SCHEMA)]
    for t in LOAD_ORDER
}

summary = []
for t in LOAD_ORDER:
    path = os.path.join(DATA_DIR, f"{t}.csv")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    summary.append({
        "table": t,
        "csv_rows": len(df),
        "csv_columns": list(df.columns),
        "db_columns_ok": set(db_columns[t]).issubset(df.columns),
    })

pd.DataFrame(summary)

,table,csv_rows,csv_columns,db_columns_ok
0,categories,30,"[category_id, category_name]",True
1,suppliers,200,"[supplier_id, country]",True
2,stores,100,"[store_id, city]",True
3,promotions,50,"[promotion_id, discount]",True
4,customers,50000,"[customer_id, city, signup_date]",True
5,employees,1000,"[employee_id, store_id, salary]",True
6,products,10000,"[product_id, category_id, supplier_id, price]",True
7,orders,300000,"[order_id, customer_id, store_id, order_date, ...",True
8,order_items,600000,"[order_item_id, order_id, product_id, qty, price]",True
9,payments,300000,"[payment_id, order_id, amount]",True


In [35]:
print(pd.read_csv(os.path.join(DATA_DIR, "orders.csv")).head())
print()
print(pd.read_csv(os.path.join(DATA_DIR, "order_items.csv")).head())

   order_id  customer_id  store_id  order_date  promotion_id
0         1        45308        33  2021-08-26            24
1         2        10070        81  2022-03-19             3
2         3        43308        17  2021-01-21            25
3         4        47997        85  2021-01-16            48
4         5        36546        81  2022-09-14            33

   order_item_id  order_id  product_id  qty  price
0              1    145042         472    3    176
1              2    110932        1666    2   1034
2              3    269799        8616    4   2290
3              4    298741        9909    3   1555
4              5    105218        1179    3    936


In [36]:
DATE_COLUMNS = {
    "customers": ["signup_date"],
    "orders": ["order_date"],
}
CHUNK = 5000   # 5000 rows x 5 columns stays below the 65,535 parameter limit of 'multi' inserts

def load_table(table):
    path = os.path.join(DATA_DIR, f"{table}.csv")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    # keep only the table's columns, in the table's order
    df = df[db_columns[table]]

    # IDs and counts -> nullable integer
    for col in df.columns:
        if col.endswith("_id") or col in ("qty", "discount"):
            df[col] = df[col].astype("Int64")

    # dates -> python date
    for col in DATE_COLUMNS.get(table, []):
        df[col] = pd.to_datetime(df[col]).dt.date

    start = time.time()
    df.to_sql(
        name=table,
        con=engine,
        schema=SCHEMA,
        if_exists="append",     # required by the brief: keep our DDL table
        index=False,
        chunksize=CHUNK,
        method="multi",
    )
    return len(df), time.time() - start

In [37]:
RESET_TABLES = False   # set to True to empty all tables first

if RESET_TABLES:
    with engine.begin() as conn:
        conn.execute(text(
            "TRUNCATE TABLE " + ", ".join(f"{SCHEMA}.{t}" for t in LOAD_ORDER) + " CASCADE"
        ))
    print("All tables truncated.")
else:
    print("Skipped.")

Skipped.


In [38]:
import sys, pandas, sqlalchemy
print(sys.executable)
print("pandas    :", pandas.__version__, pandas.__file__)
print("sqlalchemy:", sqlalchemy.__version__, sqlalchemy.__file__)

c:\Users\Admin\anaconda3\python.exe
pandas    : 3.0.5 C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\pandas\__init__.py
sqlalchemy: 2.0.31 C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sqlalchemy\__init__.py


In [39]:
import sys
!{sys.executable} -m pip install --user --upgrade "sqlalchemy>=2.0.36"


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import io

def load_table(table):
    path = os.path.join(DATA_DIR, f"{table}.csv")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df[db_columns[table]]

    for col in df.columns:
        if col.endswith("_id") or col in ("qty", "discount"):
            df[col] = df[col].astype("Int64")
    for col in DATE_COLUMNS.get(table, []):
        df[col] = pd.to_datetime(df[col]).dt.date

    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)

    start = time.time()
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.copy_expert(
            f"COPY {SCHEMA}.{table} ({', '.join(df.columns)}) FROM STDIN WITH (FORMAT csv)",
            buf,
        )
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    return len(df), time.time() - start

In [43]:
expected = {
    "categories": 30, "suppliers": 200, "stores": 100, "promotions": 50,
    "customers": 50000, "employees": 1000, "products": 10000,
    "orders": 300000, "order_items": 600000, "payments": 300000,
    "shipments": 300000, "returns": 30000,
}

rows = []
with engine.connect() as conn:
    for t in LOAD_ORDER:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA}.{t}")).scalar()
        rows.append({"table": t, "in_database": n, "expected": expected[t], "match": n == expected[t]})

check = pd.DataFrame(rows)
print("Total rows in database:", f"{check.in_database.sum():,}  (expected 1,591,380)")
check

Total rows in database: 0  (expected 1,591,380)


,table,in_database,expected,match
0,categories,0,30,False
1,suppliers,0,200,False
2,stores,0,100,False
3,promotions,0,50,False
4,customers,0,50000,False
5,employees,0,1000,False
6,products,0,10000,False
7,orders,0,300000,False
8,order_items,0,600000,False
9,payments,0,300000,False


In [44]:
fk_checks = {
    "products.category_id -> categories":   "SELECT COUNT(*) FROM products p LEFT JOIN categories c ON c.category_id=p.category_id WHERE c.category_id IS NULL",
    "products.supplier_id -> suppliers":    "SELECT COUNT(*) FROM products p LEFT JOIN suppliers s ON s.supplier_id=p.supplier_id WHERE s.supplier_id IS NULL",
    "employees.store_id -> stores":         "SELECT COUNT(*) FROM employees e LEFT JOIN stores s ON s.store_id=e.store_id WHERE s.store_id IS NULL",
    "orders.customer_id -> customers":      "SELECT COUNT(*) FROM orders o LEFT JOIN customers c ON c.customer_id=o.customer_id WHERE c.customer_id IS NULL",
    "orders.store_id -> stores":            "SELECT COUNT(*) FROM orders o LEFT JOIN stores s ON s.store_id=o.store_id WHERE s.store_id IS NULL",
    "orders.promotion_id -> promotions":    "SELECT COUNT(*) FROM orders o LEFT JOIN promotions p ON p.promotion_id=o.promotion_id WHERE o.promotion_id IS NOT NULL AND p.promotion_id IS NULL",
    "order_items.order_id -> orders":       "SELECT COUNT(*) FROM order_items i LEFT JOIN orders o ON o.order_id=i.order_id WHERE o.order_id IS NULL",
    "order_items.product_id -> products":   "SELECT COUNT(*) FROM order_items i LEFT JOIN products p ON p.product_id=i.product_id WHERE p.product_id IS NULL",
    "payments.order_id -> orders":          "SELECT COUNT(*) FROM payments y LEFT JOIN orders o ON o.order_id=y.order_id WHERE o.order_id IS NULL",
    "shipments.order_id -> orders":         "SELECT COUNT(*) FROM shipments s LEFT JOIN orders o ON o.order_id=s.order_id WHERE o.order_id IS NULL",
    "returns.order_item_id -> order_items": "SELECT COUNT(*) FROM returns r LEFT JOIN order_items i ON i.order_item_id=r.order_item_id WHERE i.order_item_id IS NULL",
}

out = []
with engine.connect() as conn:
    for name, sql in fk_checks.items():
        out.append({"foreign key": name, "orphan_rows": conn.execute(text(sql)).scalar()})
pd.DataFrame(out)

,foreign key,orphan_rows
0,products.category_id -> categories,0
1,products.supplier_id -> suppliers,0
2,employees.store_id -> stores,0
3,orders.customer_id -> customers,0
4,orders.store_id -> stores,0
5,orders.promotion_id -> promotions,0
6,order_items.order_id -> orders,0
7,order_items.product_id -> products,0
8,payments.order_id -> orders,0
9,shipments.order_id -> orders,0


In [45]:
dq = {
    "Orders with no order_items": "SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.order_id=o.order_id)",
    "Orders dated before customer signup_date": "SELECT COUNT(*) FROM orders o JOIN customers c ON c.customer_id=o.customer_id WHERE o.order_date < c.signup_date",
    "Orders with NULL promotion_id": "SELECT COUNT(*) FROM orders WHERE promotion_id IS NULL",
    "Order lines with more than one return": "SELECT COUNT(*) FROM (SELECT order_item_id FROM returns GROUP BY order_item_id HAVING COUNT(*)>1) x",
}
out = []
with engine.connect() as conn:
    for name, sql in dq.items():
        out.append({"check": name, "result": conn.execute(text(sql)).scalar()})
pd.DataFrame(out)

,check,result
0,Orders with no order_items,0
1,Orders dated before customer signup_date,0
2,Orders with NULL promotion_id,0
3,Order lines with more than one return,0


In [47]:
raw = engine.raw_connection()
try:
    print(pd.read_sql_query("""
        SELECT MIN(order_date) AS first_order, MAX(order_date) AS last_order,
               (SELECT MIN(signup_date) FROM customers) AS first_signup,
               (SELECT MAX(signup_date) FROM customers) AS last_signup
        FROM orders
    """, raw))
finally:
    raw.close()

  first_order last_order first_signup last_signup
0        None       None         None        None


C:\Users\Admin\AppData\Local\Temp\ipykernel_7596\2541492709.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query("""


In [49]:
engine.dispose()
print("Connection closed.")

Connection closed.


In [50]:
with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())
    print("User    :", conn.execute(text("SELECT current_user")).scalar())
    for t in LOAD_ORDER:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA}.{t}")).scalar()
        print(f"{t:<12} {n:>9,}")

Database: 202618057
User    : 202618057
categories           0
suppliers            0
stores               0
promotions           0
customers            0
employees            0
products             0
orders               0
order_items          0
payments             0
shipments            0
returns              0


In [51]:
# 1. Empty all tables (safe, keeps the structure)
with engine.begin() as conn:
    conn.execute(text(
        "TRUNCATE TABLE " + ", ".join(f"{SCHEMA}.{t}" for t in LOAD_ORDER) + " CASCADE"
    ))
print("Tables emptied.\n")

# 2. Load every table and confirm the count straight away
for t in LOAD_ORDER:
    n, secs = load_table(t)
    with engine.connect() as conn:
        in_db = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA}.{t}")).scalar()
    print(f"{t:<12} read {n:>9,}   in database {in_db:>9,}   ({secs:.1f} s)")

Tables emptied.

categories   read        30   in database        30   (0.0 s)
suppliers    read       200   in database       200   (0.1 s)
stores       read       100   in database       100   (0.0 s)
promotions   read        50   in database        50   (0.0 s)
customers    read    50,000   in database    50,000   (0.2 s)
employees    read     1,000   in database     1,000   (0.1 s)
products     read    10,000   in database    10,000   (0.3 s)
orders       read   300,000   in database   300,000   (10.2 s)
order_items  read   600,000   in database   600,000   (14.5 s)
payments     read   300,000   in database   300,000   (4.8 s)
shipments    read   300,000   in database   300,000   (5.1 s)
returns      read    30,000   in database    30,000   (1.0 s)
